## 01 - Ingestão de Dados Brutos (Landing Zone)
### MVP-3 — Engenharia de Dados
 
  Este notebook realiza a ingestão dos arquivos originais utilizados
  no projeto.
 
#### Fontes
- IMDb Dataset of 50K Movie Reviews (https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)
- Rotten Tomatoes Movies and Critic Reviews Dataset ( https://www.kaggle.com/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset)
 
Obs: Nenhuma transformação dos dados é realizada neste notebook.

In [0]:
#Instalando lib kagglehub
%pip install -q kagglehub

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#restartando o kernel, conforme orientacao
dbutils.library.restartPython()

In [0]:
import os
import shutil
import kagglehub


# ============================================================
# DATASETS KAGGLE
# ============================================================

DATASET_IMDB = (
    "lakshmi25npathi/"
    "imdb-dataset-of-50k-movie-reviews"
)

DATASET_RT = (
    "stefanoleone992/"
    "rotten-tomatoes-movies-and-critic-reviews-dataset"
)


# ============================================================
# ARQUIVOS ESPERADOS
# ============================================================

NOME_ARQUIVO_IMDB = "IMDB Dataset.csv"

NOME_ARQUIVO_RT_FILMES = "rotten_tomatoes_movies.csv"

NOME_ARQUIVO_RT_AVALIACOES = (
    "rotten_tomatoes_critic_reviews.csv"
)


# ============================================================
# LANDING ZONE
# ============================================================

CAMINHO_VOLUME = (
    "/Volumes/mvp3_eng_dados/"
    "dados_brutos/"
    "volume_dados_brutos"
)

CAMINHO_IMDB = f"{CAMINHO_VOLUME}/imdb"

CAMINHO_RT = f"{CAMINHO_VOLUME}/rotten_tomatoes"


print("✓ Configuração carregada")
print(f"Landing Zone: {CAMINHO_VOLUME}")

✓ Configuração carregada
Landing Zone: /Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos


In [0]:
os.makedirs(
    CAMINHO_IMDB,
    exist_ok=True
)

os.makedirs(
    CAMINHO_RT,
    exist_ok=True
)


print("✓ Diretórios da Landing Zone preparados")

✓ Diretórios da Landing Zone preparados


In [0]:
# ============================================================
# DOWNLOAD DOS DATASETS
# ============================================================

print("Baixando dataset IMDb...")

caminho_cache_imdb = kagglehub.dataset_download(
    DATASET_IMDB
)

print("✓ IMDb disponível")


print("\nBaixando dataset Rotten Tomatoes...")

caminho_cache_rt = kagglehub.dataset_download(
    DATASET_RT
)

print("✓ Rotten Tomatoes disponível")


# ============================================================
# CAMINHOS DOS ARQUIVOS BAIXADOS
# ============================================================

arquivo_imdb_local = os.path.join(
    caminho_cache_imdb,
    NOME_ARQUIVO_IMDB
)

arquivo_rt_filmes_local = os.path.join(
    caminho_cache_rt,
    NOME_ARQUIVO_RT_FILMES
)

arquivo_rt_avaliacoes_local = os.path.join(
    caminho_cache_rt,
    NOME_ARQUIVO_RT_AVALIACOES
)


# ============================================================
# VALIDAÇÃO
# ============================================================

assert os.path.isfile(arquivo_imdb_local), (
    f"IMDb não encontrado: {arquivo_imdb_local}"
)

assert os.path.isfile(arquivo_rt_filmes_local), (
    f"RT Filmes não encontrado: {arquivo_rt_filmes_local}"
)

assert os.path.isfile(arquivo_rt_avaliacoes_local), (
    f"RT Avaliações não encontrado: "
    f"{arquivo_rt_avaliacoes_local}"
)


print("\nArquivos localizados:")
print("✓ IMDB Dataset.csv")
print("✓ rotten_tomatoes_movies.csv")
print("✓ rotten_tomatoes_critic_reviews.csv")

Baixando dataset IMDb...
✓ IMDb disponível

Baixando dataset Rotten Tomatoes...
✓ Rotten Tomatoes disponível

Arquivos localizados:
✓ IMDB Dataset.csv
✓ rotten_tomatoes_movies.csv
✓ rotten_tomatoes_critic_reviews.csv


In [0]:
# ============================================================
# DESTINOS
# ============================================================

DESTINO_IMDB = os.path.join(
    CAMINHO_IMDB,
    NOME_ARQUIVO_IMDB
)

DESTINO_RT_FILMES = os.path.join(
    CAMINHO_RT,
    NOME_ARQUIVO_RT_FILMES
)

DESTINO_RT_AVALIACOES = os.path.join(
    CAMINHO_RT,
    NOME_ARQUIVO_RT_AVALIACOES
)


# ============================================================
# CÓPIA PARA A LANDING ZONE
# ============================================================

shutil.copy2(
    arquivo_imdb_local,
    DESTINO_IMDB
)

shutil.copy2(
    arquivo_rt_filmes_local,
    DESTINO_RT_FILMES
)

shutil.copy2(
    arquivo_rt_avaliacoes_local,
    DESTINO_RT_AVALIACOES
)


print("✓ Arquivos persistidos na Landing Zone")

✓ Arquivos persistidos na Landing Zone


In [0]:
arquivos_persistidos = [
    DESTINO_IMDB,
    DESTINO_RT_FILMES,
    DESTINO_RT_AVALIACOES
]


for arquivo in arquivos_persistidos:

    assert os.path.isfile(arquivo), (
        f"Arquivo não encontrado: {arquivo}"
    )

    assert os.path.getsize(arquivo) > 0, (
        f"Arquivo vazio: {arquivo}"
    )

    tamanho_mb = os.path.getsize(arquivo) / (1024 * 1024)

    print(
        f"✓ {os.path.basename(arquivo)} "
        f"({tamanho_mb:.2f} MB)"
    )

✓ IMDB Dataset.csv (63.14 MB)
✓ rotten_tomatoes_movies.csv (16.32 MB)
✓ rotten_tomatoes_critic_reviews.csv (215.58 MB)


In [0]:
def ler_csv(caminho):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(caminho)
    )


df_imdb_teste = ler_csv(
    DESTINO_IMDB
)

df_rt_filmes_teste = ler_csv(
    DESTINO_RT_FILMES
)

df_rt_avaliacoes_teste = ler_csv(
    DESTINO_RT_AVALIACOES
)


print("✓ Os três arquivos foram lidos pelo Spark")

✓ Os três arquivos foram lidos pelo Spark


In [0]:
print("=" * 60)
print("IMDb")
print("=" * 60)

df_imdb_teste.printSchema()


print("\n" + "=" * 60)
print("ROTTEN TOMATOES - FILMES")
print("=" * 60)

df_rt_filmes_teste.printSchema()


print("\n" + "=" * 60)
print("ROTTEN TOMATOES - AVALIAÇÕES")
print("=" * 60)

df_rt_avaliacoes_teste.printSchema()

IMDb
root
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)


ROTTEN TOMATOES - FILMES
root
 |-- rotten_tomatoes_link: string (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- movie_info: string (nullable = true)
 |-- critics_consensus: string (nullable = true)
 |-- content_rating: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- actors: string (nullable = true)
 |-- original_release_date: string (nullable = true)
 |-- streaming_release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- production_company: string (nullable = true)
 |-- tomatometer_status: string (nullable = true)
 |-- tomatometer_rating: string (nullable = true)
 |-- tomatometer_count: string (nullable = true)
 |-- audience_status: string (nullable = true)
 |-- audience_rating: string (nullable = true)
 |-- audience_count: string (nullable = true)
 |-- t

In [0]:
# ============================================================
# COLUNAS ESPERADAS
# ============================================================

COLUNAS_IMDB_ESPERADAS = {
    "review",
    "sentiment"
}

COLUNAS_RT_FILMES_MINIMAS = {
    "rotten_tomatoes_link",
    "movie_title"
}

COLUNAS_RT_AVALIACOES_ESPERADAS = {
    "rotten_tomatoes_link",
    "critic_name",
    "top_critic",
    "publisher_name",
    "review_type",
    "review_score",
    "review_date",
    "review_content"
}


# ============================================================
# VALIDAÇÃO
# ============================================================

assert COLUNAS_IMDB_ESPERADAS.issubset(
    set(df_imdb_teste.columns)
), "IMDb: estrutura diferente da esperada."

assert COLUNAS_RT_FILMES_MINIMAS.issubset(
    set(df_rt_filmes_teste.columns)
), "RT Filmes: estrutura diferente da esperada."

assert COLUNAS_RT_AVALIACOES_ESPERADAS.issubset(
    set(df_rt_avaliacoes_teste.columns)
), "RT Avaliações: estrutura diferente da esperada."


print("✓ Estrutura IMDb validada")
print("✓ Estrutura RT Filmes validada")
print("✓ Estrutura RT Avaliações validada")

✓ Estrutura IMDb validada
✓ Estrutura RT Filmes validada
✓ Estrutura RT Avaliações validada


In [0]:
print("IMDb")
display(
    df_imdb_teste.limit(5)
)


print("Rotten Tomatoes - Filmes")
display(
    df_rt_filmes_teste.limit(5)
)


print("Rotten Tomatoes - Avaliações")
display(
    df_rt_avaliacoes_teste.limit(5)
)

IMDb


review,sentiment
"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.",positive
"A wonderful little production. The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. The actors are extremely well chosen- Michael Sheen not only ""has got all the polari"" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.",positive
"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk Addiction, I thought it was proof that Woody Allen is still fully in control of the style many of us have grown to love.This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never been impressed with Scarlet Johanson, in this she managed to tone down her ""sexy"" image and jumped right into a average, but spirited young woman.This may not be the crown jewel of his career, but it was wittier than ""Devil Wears Prada"" and more interesting than ""Superman"" a great comedy to go see with friends.",positive
"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the mo

Rotten Tomatoes - Filmes


rotten_tomatoes_link,movie_title,movie_info,critics_consensus,content_rating,genres,directors,authors,actors,original_release_date,streaming_release_date,runtime,production_company,tomatometer_status,tomatometer_rating,tomatometer_count,audience_status,audience_rating,audience_count,tomatometer_top_critics_count,tomatometer_fresh_critics_count,tomatometer_rotten_critics_count
m/0814255,Percy Jackson & the Olympians: The Lightning Thief,"Always trouble-prone, the life of teenager Percy Jackson (Logan Lerman) gets a lot more complicated when he learns he's the son of the Greek god Poseidon. At a training ground for the children of deities, Percy learns to harness his divine powers and prepare for the adventure of a lifetime: he must prevent a feud among the Olympians from erupting into a devastating war on Earth, and rescue his mother from the clutches of Hades, god of the underworld.","Though it may seem like just another Harry Potter knockoff, Percy Jackson benefits from a strong supporting cast, a speedy plot, and plenty of fun with Greek mythology.",PG,"Action & Adventure, Comedy, Drama, Science Fiction & Fantasy",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Daddario, Jake Abel, Sean Bean, Pierce Brosnan, Steve Coogan, Rosario Dawson, Melina Kanakaredes, Catherine Keener, Kevin Mckidd, Joe Pantoliano, Uma Thurman, Ray Winstone, Julian Richings, Bonita Friedericy, Annie Ilonzeh, Tania Saulnier, Marie Avgeropoulos, Luisa D'Oliveira, Christie Laing, Marielle Jaffe, Elisa King, Chrystal Tisiga, Alexis Knapp, Charlie Gallant, Chelan Simmons, Andrea Brooks, Natassia Malthe, Max Van Ville, Serinda Swan, Dimitri Lekkos, Ona Grauer, Stefanie von Pfetten, Conrad Coates, Erica Cerra, Dylan Neal, Luke Camilleri, Holly Hougham, Ina Geraldine, Raquel Riskin, Yusleidis Oquendo, Janine Edwards, Valerie Tian, Violet Columbus, Sarah Smyth, Merritt Patterson, Julie Luck, Andrea Day, John Stewart, Suzanne Ristic, Deejay Jackson, Matthew Garrick, Stan Carp, Suzanna Ristic, Richard Harmon, Maria Olsen, Robin Lemon, Doyle Devereux, Tom Pickett, VJ Delos-Reyes, Tim Aas, Keith Dallas, Spencer Atkinson, Maya Washington, Loyd Bateman, Victor Ayala, Zane Holtz, Eli Zagoudakis, Matt Reimer, Rob Hayter, Lloyd Bateman, Shawn Beaton, Jarod Joseph, Reilly Dolman, Paul Cummings, Julie Brar, Dejan Loyola, Damian Arman, Mario Casoria, Dorla Bell, Carolyn Adair (II), Jade Pawluk, G. Patrick Currie, Darian Arman, Mariela Zapata, David L. Smith",2010-02-12,2015-11-25,119,20th Century Fox,Rotten,49,149,Spilled,53,254421,43,73,76
m/0878835,Please Give,"Kate (Catherine Keener) and her husband Alex (Oliver Platt) are wealthy New Yorkers who prowl estate sales and make a tidy profit reselling items they bought cheaply. They buy the apartment next door and plan to remodel just as soon as its current occupant, a cranky old woman, dies. Kate is troubled by the way she and her husband earn a living, and tries to assuage her guilt by befriending her tenant and the woman's granddaughters, but her overtures lead to unexpected consequences.","Nicole Holofcener's newest might seem slight in places, but its rendering of complex characters in a conflicted economic landscape is varied, natural, and touching all the same.",R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, Rebecca Hall, Sarah Steele, Ann Morgan Guilbert, Kevin Corrigan, Lois Smith",2010-04-30,2012-09-04,90,Sony Pictures Classics,Certified-Fresh,87,142,Upright,64,11574,44,123,19
m/10,10,"A successful, middle-aged Hollywood songwriter falls hopelessly in love with the woman of his dreams, and even follows the girl and her new husband to their Mexican honeymoon resort. While his behavior seems sure to land him in trouble, out of the blue fate plays into his hands.","Blake Edwards' bawdy comedy may not score a perfect 10, but Dudley Moore's self-deprecating performance makes this midlife crisis persistently funny.",R,"C

Rotten Tomatoes - Avaliações


rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_type,review_score,review_date,review_content
m/0814255,Andrew L. Urban,False,Urban Cinefile,Fresh,null,2010-02-06,A fantasy adventure that fuses Greek mythology to contemporary American places and values. Anyone around 15 (give or take a couple of years) will thrill to the visual spectacle
m/0814255,Louise Keller,False,Urban Cinefile,Fresh,null,2010-02-06,"Uma Thurman as Medusa, the gorgon with a coiffure of writhing snakes and stone-inducing hypnotic gaze is one of the highlights of this bewitching fantasy"
m/0814255,null,False,FILMINK (Australia),Fresh,null,2010-02-09,"With a top-notch cast and dazzling special effects, this will tide the teens over until the next Harry Potter instalment."
m/0814255,Ben McEachen,False,Sunday Mail (Australia),Fresh,3.5/5,2010-02-09,"Whether audiences will get behind The Lightning Thief is hard to predict. Overall, it's an entertaining introduction to a promising new world -- but will the consuming shadow of Potter be too big to break free of?"
m/0814255,Ethan Alter,True,Hollywood Reporter,Rotten,null,2010-02-10,"What's really lacking in The Lightning Thief is a genuine sense of wonder, the same thing that brings viewers back to Hogwarts over and over again."


In [0]:
print("=" * 60)
print("INGESTÃO DE DADOS BRUTOS CONCLUÍDA")
print("=" * 60)


print("\nLanding Zone:")
print(CAMINHO_VOLUME)


print("\nArquivos ingeridos:")
print(f"✓ {NOME_ARQUIVO_IMDB}")
print(f"✓ {NOME_ARQUIVO_RT_FILMES}")
print(f"✓ {NOME_ARQUIVO_RT_AVALIACOES}")


print("\nValidações:")
print("✓ Datasets obtidos do Kaggle")
print("✓ Arquivos originais localizados")
print("✓ Arquivos persistidos no Volume")
print("✓ Leitura pelo Spark validada")
print("✓ Estrutura mínima validada")


print("\nTransformações realizadas:")
print("NENHUMA")


print("\nStatus:")
print("✓ LANDING ZONE VALIDADA")


print("\nPróxima etapa:")
print("02_camada_bronze")

INGESTÃO DE DADOS BRUTOS CONCLUÍDA

Landing Zone:
/Volumes/mvp3_eng_dados/dados_brutos/volume_dados_brutos

Arquivos ingeridos:
✓ IMDB Dataset.csv
✓ rotten_tomatoes_movies.csv
✓ rotten_tomatoes_critic_reviews.csv

Validações:
✓ Datasets obtidos do Kaggle
✓ Arquivos originais localizados
✓ Arquivos persistidos no Volume
✓ Leitura pelo Spark validada
✓ Estrutura mínima validada

Transformações realizadas:
NENHUMA

Status:
✓ LANDING ZONE VALIDADA

Próxima etapa:
02_camada_bronze
